# 03 Expert review

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CoralsINPA/RedSeaCoralIdentifier/blob/main/notebooks/03_expert_review.ipynb)

No GPU is needed. Run `02_zero_shot.ipynb` first: this notebook reads the baseline it wrote. Run the first cell, then the others in order.

**Safe to run and re-run.** It reads the shared Drive and only creates new files, never overwriting one.

| Step | What it does | Time | Reads | Writes |
|---|---|---|---|---|
| Setup | mounts Drive, updates the clone, installs `coralid[viz]` | 1-3 min | GitHub | nothing on Drive |
| Config | loads the run, the baseline and the manifest | seconds | `<run>/predictions/...__all.parquet`, `<run>/manifest/manifest.parquet` | nothing |
| Portrait impact, expert set | every expert row on a portrait photo: label, legacy prediction, new prediction | seconds | `assaf_expert_test_set.csv` | `<run>/reports/portrait_impact_<date>/` (a new folder) |
| Blind review sheet (**on hold**) | only with `RUN_EXPORT = True`: prints which predictions it uses, then a random, mixed, shuffled sheet with no model output; a subset for Assaf; the hidden key | 2-5 min | the baseline, photos, mask JSONs | `<run>/reviews/exports/` (sheets and PDFs to hand out) and `<run>/reviews/keys/` (the answer key, **do not share**) |
| Scoring | after the reviews come back: checks the key's predictions are still the current baseline, then precision, miss rate and recall with intervals, analyst against expert | seconds | `<run>/reviews/completed/` | `<run>/reports/blind_score_<name>_<date>/` (a new folder) |

**Why the portrait rows matter.** The legacy code cut the wrong region on the 221 portrait photos (1920x2560 frames). Assaf's expert labels were judged on crops cut by that code. For portrait photos the label may describe a different-looking crop than the one `coralid` cuts now, so those rows are listed and flagged.

**There is no valid Millepora precision figure yet.** An earlier check was made on the predictions of a pipeline that is no longer used, by a non-expert who saw the model's answer; it is discarded. The blind review below redoes it on the 20-genus baseline.

**The blind review draws only from the 20-genus baseline**, `<run>/predictions/finetuned-bioclip__taxonomy_v1__hard20__all.parquet`. Before sampling it reads that file's `.meta.json` and stops unless the label set is `hard20` and the model and prompt set are the config's. The file, its commit and creation time, the model, the prompt set and the label set are printed and recorded in the answer key; scoring stops if the baseline has been made again since.

**The blind review export (the section The blind, mixed review sheet) is on hold.** The blind review will be combined with the larger expert set (`docs/algorithm_ideas.md` idea 1.1) so that Assaf reviews once. The cell is kept and works, but it writes nothing unless you set `RUN_EXPORT = True` in it. The scoring cell then waits for completed sheets.

**What to paste back to the next session:** the printed counts of the portrait-impact cell, the source of the review (when exported), and (after the reviews) the scoring.


In [ ]:
import os
import subprocess
import sys

# ================================ SETTINGS: the only place to edit ================================
DATA_ROOT = "/content/drive/MyDrive/colab notebooks/CoralsMasking"  # the Drive folder CoralsMasking
BRANCH = "main"
REPO_URL = "https://github.com/CoralsINPA/RedSeaCoralIdentifier"
REPO_DIR = "/content/RedSeaCoralIdentifier"
EXTRAS = "viz"
# ==================================================================================================

# The package cannot install itself before it exists. A clone left by an earlier session of this
# runtime is brought to the branch head BEFORE anything is imported from it.
if os.path.isdir(f"{REPO_DIR}/.git"):
    steps = [
        ["git", "-C", REPO_DIR, "fetch", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "checkout", BRANCH],
        ["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "submodule", "update", "--init"],
    ]
else:
    steps = [["git", "clone", "--recurse-submodules", "--branch", BRANCH, REPO_URL, REPO_DIR]]
for step in steps:
    print("$", " ".join(step))
    subprocess.run(step, check=True)
sys.path.insert(0, f"{REPO_DIR}/src")  # enough to import coralid.colab (standard library only)

from coralid.colab import setup  # noqa: E402

env = setup(
    EXTRAS, data_root=DATA_ROOT, branch=BRANCH, repo_url=REPO_URL, repo_dir=REPO_DIR, update=False
)

In [ ]:
import json
import logging

import pandas as pd

from coralid.baseline import baseline_paths
from coralid.config import load_config
from coralid.manifest import read_manifest

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("coralid").setLevel(logging.INFO)

cfg = load_config(f"{REPO_DIR}/configs/iou60_sta55_pts32.yaml")  # reads CORALID_DATA_ROOT
GENUS = "Millepora"
EXPERT_CSV = cfg.run_dir / "assaf_expert_test_set.csv"
paths = baseline_paths(cfg)
for label, path in {
    "manifest": cfg.manifest_path,
    "baseline": paths.predictions,
    "expert set": EXPERT_CSV,
}.items():
    print(f"  {label:16} {'ok     ' if path.exists() else 'MISSING'} {path}")
if not paths.predictions.exists():
    raise FileNotFoundError("The baseline is missing: run notebooks/02_zero_shot.ipynb first.")

manifest = read_manifest(cfg.manifest_path)
predictions = pd.read_parquet(paths.predictions)
print(f"{len(predictions)} baseline crops on {predictions['image_name'].nunique()} photos")

## Portrait impact on the expert set

Every row of `assaf_expert_test_set.csv` on a portrait photo, with Assaf's label and status, the prediction stored in the CSV (legacy) and the new baseline's. The summary says how many of the "unidentifiable" rows are among them, which separates "mis-cut" from "blurred" as the reason for an unidentifiable call.

In [ ]:
from coralid.evaluation import load_expert_set, load_legacy_predictions
from coralid.portrait_impact import expert_portrait_impact, write_portrait_report

expert = load_expert_set(EXPERT_CSV, cfg.zero_shot.label_set)
legacy_predictions = load_legacy_predictions(EXPERT_CSV)
expert_rows, expert_summary = expert_portrait_impact(
    expert, legacy_predictions, predictions, manifest
)

s = expert_summary
print(f"expert rows: {s['n_expert_rows']}, by orientation {s['rows_by_orientation']}")
print(
    f"on portrait photos: {s['portrait_rows']} rows = {s['portrait_share_of_expert_rows']:.1%} "
    f"of the expert rows (portrait photos are {s['portrait_share_of_photos']:.1%} of all photos)"
)
print(
    f"unidentifiable: {s['unidentifiable_on_portrait']} of {s['unidentifiable_total']} "
    "unidentifiable rows are on portrait photos"
)
print("status by orientation:", json.dumps(s["status_by_orientation"]))
print(
    "portrait rows: in the new baseline",
    s["portrait_in_new_baseline"],
    "| prediction changed",
    s["portrait_prediction_changed"],
)
print(
    "portrait rows scored (named genus): legacy", s["portrait_legacy"], "| new", s["portrait_new"]
)
print(
    "landscape rows whose prediction changed:",
    s["landscape_prediction_changed"],
    "(close to 0 if the CSV's predictions came from this configuration)",
)
print("\n" + s["caution"] + "\n")
display(expert_rows.drop(columns=["key", "expert_raw"]))
report_folder = write_portrait_report(expert_rows, expert_summary, cfg.reports_dir)
print("Saved (a new folder, nothing was overwritten):", report_folder)

## The blind, mixed review sheet (on hold)

**On hold:** the user will hand out one combined review later (this Millepora sheet together with the larger expert set), so the cell below does nothing until `RUN_EXPORT = True`.

A random sample, across all confidence levels, of the crops the 20-genus baseline calls Millepora, plus about as many it did not, shuffled, **with no prediction, confidence or runner-up shown**. About 40 of the crops go on a second sheet for Assaf, to measure how often the analyst agrees with him. Every sampled crop must be a kept row of the manifest.

The cell first prints **which predictions it draws from** (the file, the commit and time of the baseline job, the model, the prompt set and the label set) and stops if they are not the `hard20` baseline of the config. The same record goes into the answer key.

Hand out the files in `<run>/reviews/exports/` (a CSV and a PDF per person). **Keep `<run>/reviews/keys/` to yourself.** Reviewers fill `is_millepora` (yes / no / unsure) and, for every `no`, `actual_genus`; completed CSVs go into `<run>/reviews/completed/` under the same file names. Run this cell only once per seed: a second run with the same seed on the same day refuses, as it would overwrite.

In [ ]:
from coralid.genus_audit import build_blind_review
from coralid.masks import AnnotationStore
from coralid.review_export import write_blind_review
from coralid.review_source import baseline_source

# ================================ ON HOLD ================================
# The blind review is to be combined with the larger expert set, so that Assaf reviews once.
# Keep this False until then: with False the cell writes nothing.
RUN_EXPORT = False
# ==========================================================================
N_PRED, N_OTHER, N_ASSAF, SEED = 150, 150, 40, 0  # crops called Millepora, others, for Assaf

blind_name = None  # the name of the review, once exported (the scoring cell needs it)
if RUN_EXPORT:
    source = baseline_source(cfg)  # stops unless this is the hard20 baseline of the config
    print("The review is drawn only from:")
    print("\n".join(source.lines()))
    review = build_blind_review(
        predictions,
        manifest,
        GENUS,
        n_pred=N_PRED,
        n_other=N_OTHER,
        n_assaf=N_ASSAF,
        seed=SEED,
        source=source,
    )
    print(json.dumps(review.meta, indent=1, default=str))
    files = write_blind_review(
        review,
        cfg.reviews_dir / "exports",
        cfg.reviews_dir / "keys",
        AnnotationStore(cfg.json_dir),
        cfg.images_dir,
        crops=cfg.crops,
    )
    blind_name = files["key"].name.removesuffix("_answer_key.csv")
    print("\nname of this review (needed for the scoring):", blind_name)
    for what, file in files.items():
        print(f"  {what:12} {file}")
else:
    print("The blind review export is on hold (RUN_EXPORT = False): nothing was written.")

## Scoring the completed reviews

After the analyst (and Assaf, for his subset) have returned their sheets, put them in `<run>/reviews/completed/` named `<name>_analyst.csv` and `<name>_assaf.csv`. This cell ingests them by key, lists anything to fix (a `no` without an `actual_genus`, an unreadable answer, a genus name it does not know), and scores them against the hidden key (after checking that the key's predictions are still the current baseline): precision of the baseline's Millepora calls, the miss rate among its other calls and the recall, with intervals and with the unsure counted both ways; the precision by confidence band; how often the analyst agrees with Assaf; and what the false positives really are. Until the sheets are there it only says what it is waiting for. With the export on hold, set `SCORE_NAME` to the name of a review that was exported earlier.

In [ ]:
from coralid.labeling import ingest_review
from coralid.review_export import read_blind_key
from coralid.review_scoring import score_blind_review, write_blind_score
from coralid.review_source import baseline_source

SCORE_NAME = None  # later: the name printed by the export, e.g. "millepora_blind_2026-10-12_seed0"
name = SCORE_NAME or blind_name
if name is None:
    analyst_file = assaf_file = None
    print("Nothing to score yet: the export is on hold. Set SCORE_NAME once sheets are exported.")
else:
    analyst_file = cfg.reviews_dir / "completed" / f"{name}_analyst.csv"
    assaf_file = cfg.reviews_dir / "completed" / f"{name}_assaf.csv"

if analyst_file is not None and not analyst_file.exists():
    print(f"Waiting for the analyst's completed sheet: {analyst_file}")
elif analyst_file is not None:
    key, meta = read_blind_key(cfg.reviews_dir / "keys", name)
    analyst = ingest_review(analyst_file, meta["answer_column"], cfg.zero_shot.label_set)
    expert_sheet = ingest_review(assaf_file, meta["answer_column"]) if assaf_file.exists() else None
    print("analyst answers:", analyst.counts())
    if expert_sheet is not None:
        print("Assaf answers:  ", expert_sheet.counts())
    else:
        print(f"(no sheet from Assaf yet: {assaf_file})")
    if len(analyst.problems):
        print("\nTo fix in the analyst's sheet:")
        display(analyst.problems)
    current = baseline_source(cfg)
    print(
        "\nThe key was drawn from:", meta["source"]["file"], "made", meta["source"]["created_utc"]
    )
    print("The baseline now is:   ", current.file, "made", current.created_utc)
    score = score_blind_review(analyst, key, meta, expert_sheet, current_source=current)
    print(json.dumps(score.to_dict(), indent=1, default=str))
    print("\nPrecision of the baseline's Millepora calls by confidence band (the random sample):")
    display(score.precision_by_confidence.round(3))
    print("\nSaved (a new folder):", write_blind_score(score, cfg.reports_dir, name))